# KALIA mix v2 — build train.bin/val.bin from prepared shards

Reuses the tokenized shards from the completed `kalia-prep-v2` run (mounted read-only at `/kaggle/input`), so no re-tokenization is needed. Outputs the interleaved 2.4B-token `train.bin` and the proportional 10M-token `val.bin`.

In [ ]:
!mkdir -p /kaggle/working/data

In [ ]:
import glob
import subprocess

def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"{name} not found - attach the kalia-prep-v2 output"
    return hits[0]

mix = find("mix_bins.py")
shards = ",".join([
    f"{find('tinystories.bin')}:20",
    f"{find('smollm_fineweb_edu.bin')}:60",
    f"{find('cosmopedia.bin')}:15",
    f"{find('stack_smol.bin')}:5",
])
cmd = [
    "python", mix,
    "--shards", shards,
    "--val-tokens", "10000000",
    "--train-tokens", "2400000000",
    "--val-out", "/kaggle/working/data/val.bin",
    "--train-out", "/kaggle/working/data/train.bin",
    "--meta", "/kaggle/working/data/mix_meta.json",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
!ls -lh /kaggle/working/data/
!cat /kaggle/working/data/mix_meta.json

Done. The v0.2.0 training kernel will attach this run's output as a kernel source (train.bin + val.bin).